In [1]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
import nltk
import re
import pandas as pd
from scipy.sparse import hstack
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, multilabel_confusion_matrix

In [2]:
df = pd.read_csv('train.csv')

In [3]:
print(df.head(10))

      id      cuisine                                        ingredients
0  10259        greek  ['romaine lettuce', 'black olives', 'grape tom...
1  25693  southern_us  ['plain flour', 'ground pepper', 'salt', 'toma...
2  20130     filipino  ['eggs', 'pepper', 'salt', 'mayonaise', 'cooki...
3  22213       indian        ['water', 'vegetable oil', 'wheat', 'salt']
4  13162       indian  ['black pepper', 'shallots', 'cornflour', 'cay...
5   6602     jamaican  ['plain flour', 'sugar', 'butter', 'eggs', 'fr...
6  42779      spanish  ['olive oil', 'salt', 'medium shrimp', 'pepper...
7   3735      italian  ['sugar', 'pistachio nuts', 'white almond bark...
8  16903      mexican  ['olive oil', 'purple onion', 'fresh pineapple...
9  12734      italian  ['chopped tomatoes', 'fresh basil', 'garlic', ...


In [4]:
# X is ingredients input, y is cuisine output
X = df["ingredients"]
y = df["cuisine"]

# we split overall X, y into train/dev and test sets
# X,y train_overall is for the train and dev sets
X_train_overall, X_test, y_train_overall, y_test = train_test_split(X, y, random_state=67, test_size=0.30, shuffle=True)

# X,y train_overall is split into train and dev sets
X_train, X_dev, y_train, y_dev = train_test_split(X_train_overall, y_train_overall, random_state=67, test_size=0.30, shuffle=True)

print("train size: " + str(len(X_train)))
print("dev size: " + str(len(X_dev)))
print("test size: " + str(len(X_test)))
print(y_train)

train size: 9892
dev size: 4240
test size: 6057
19588         indian
5505         british
13980         korean
4950         mexican
8490         italian
            ...     
10017    southern_us
9454     southern_us
18887         french
17392       moroccan
10819        chinese
Name: cuisine, Length: 9892, dtype: str


In [5]:
vectorizer = CountVectorizer(tokenizer=lambda x: [i.replace("\"", "\'").replace("\'", "").replace("[", "").replace("]", "").strip() for i in x.split(',')])
vectorizer.fit(X_train)

print(vectorizer.get_feature_names_out())

/ix1/cs1671-2026f/class-env/.pixi/envs/default/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


['' '(    oz.) tomato sauce' '(   oz.) tomato paste' ... 'zinfandel'
 'ziti' 'zucchini']


In [6]:
features = {"name": vectorizer.get_feature_names_out()}
feature_df = pd.DataFrame(features)
feature_df.sample(30)

,name
2546,manzanilla
1764,garden peas
3788,soba noodles
885,chopped fresh mint
2148,instant white rice
2818,onion rings
1883,grated lemon peel
2169,italian seasoned dry bread crumbs
2282,kochujang
601,canned black beans


In [7]:
train_features = vectorizer.transform(X_train)
dev_features = vectorizer.transform(X_dev)
test_features = vectorizer.transform(X_test)
print(train_features)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 106419 stored elements and shape (9892, 4544)>
  Coords	Values
  (0, 234)	1
  (0, 354)	1
  (0, 638)	1
  (0, 929)	1
  (0, 940)	1
  (0, 1048)	1
  (0, 1561)	1
  (0, 2128)	1
  (0, 2290)	1
  (0, 2334)	1
  (0, 2540)	1
  (0, 2965)	1
  (0, 3734)	1
  (0, 3941)	1
  (0, 4354)	1
  (0, 4407)	1
  (1, 261)	1
  (1, 647)	1
  (1, 1382)	1
  (1, 1424)	1
  (1, 1574)	1
  (1, 2636)	1
  (1, 2714)	1
  (1, 2792)	1
  (1, 2824)	1
  :	:
  (9890, 1051)	1
  (9890, 1473)	1
  (9890, 1554)	1
  (9890, 1626)	1
  (9890, 1961)	1
  (9890, 1965)	1
  (9890, 1975)	1
  (9890, 2644)	1
  (9890, 2809)	1
  (9890, 2896)	1
  (9890, 3345)	1
  (9890, 3527)	1
  (9890, 3941)	1
  (9891, 549)	1
  (9891, 810)	1
  (9891, 1766)	1
  (9891, 2025)	1
  (9891, 2792)	1
  (9891, 3124)	1
  (9891, 3314)	1
  (9891, 3566)	1
  (9891, 3628)	1
  (9891, 3835)	1
  (9891, 3941)	1
  (9891, 4354)	1


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#baseline model
model1 = LogisticRegression()
model1.fit(train_features, y_train)
model1_dev_predictions = model1.predict(dev_features)
result1 = accuracy_score(model1_dev_predictions, y_dev)
print(result1)

0.7285377358490566


In [9]:
#one words as in separated by space not ingredient
one_word_vectorizer = CountVectorizer()
one_word_vectorizer.fit(vectorizer.get_feature_names_out())

print(one_word_vectorizer.get_feature_names_out())
features1 = {"name": one_word_vectorizer.get_feature_names_out()}
feature_df1 = pd.DataFrame(features1)
feature_df1.sample(30)

vocab_bridge_matrix = one_word_vectorizer.transform(vectorizer.get_feature_names_out())

one_word_train = train_features @ vocab_bridge_matrix
one_word_dev = dev_features @ vocab_bridge_matrix

model2 = LogisticRegression()
model2.fit(one_word_train, y_train)
model2_dev_predictions = model2.predict(one_word_dev)
result2 = accuracy_score(y_dev, model2_dev_predictions)
print(result2)

#comparing the shapes of the baseline training feature to the individual words feature
print(train_features.shape)
print(one_word_train.shape)

['00' '10' '14' ... 'ziti' 'zucchini' 'épices']
0.7358490566037735
(9892, 4544)
(9892, 2284)


In [10]:
feature_df1.sample(30)

,name
731,filling
1020,jarlsberg
713,feet
1399,ox
1618,pulp
543,crisps
2204,wafer
758,fleshed
1018,jambalaya
1388,orecchiette


In [11]:
#removing prep words
prep_words = ["chopped", "diced", "sliced", "minced", "grated", "shredded", "melted", "crushed", "ground", "organic", "fresh", "frozen"]
def clean(x):
    items = [i.replace("\"", "").replace("'", "").replace("[", "").replace("]", "").strip() for i in x.split(',')]
    return [i for i in items if not any(w in i.split() for w in prep_words)]
remove_prep_vectorizer = CountVectorizer(tokenizer=clean)
remove_prep_vectorizer.fit(X_train)

# # Check if "chopped" exists in your first vectorizer's vocabulary
# phrases_vocab = remove_prep_vectorizer.get_feature_names_out()
# has_chopped_phrases = "ground" in phrases_vocab
# print(f"Is 'chopped' in the first vectorizer's vocabulary? {has_chopped_phrases}")

remove_prep_train = remove_prep_vectorizer.transform(X_train)
remove_prep_dev = remove_prep_vectorizer.transform(X_dev)

model3 = LogisticRegression()
model3.fit(remove_prep_train, y_train)
model3_dev_predictions = model3.predict(remove_prep_dev)
result3 = accuracy_score(model3_dev_predictions, y_dev)
print(result3)

/ix1/cs1671-2026f/class-env/.pixi/envs/default/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


0.7158018867924528


In [12]:
#removing top 5 words
top_words = ["pepper", "salt", "oil", "garlic", "ground", "fresh"]
def clean(x):
    items = [i.replace("\"", "").replace("'", "").replace("[", "").replace("]", "").strip() for i in x.split(',')]
    return [i for i in items if not any(w in i.split() for w in top_words)]
remove_top_vectorizer = CountVectorizer(tokenizer=clean)
remove_top_vectorizer.fit(X_train)

remove_top_train = remove_top_vectorizer.transform(X_train)
remove_top_dev = remove_top_vectorizer.transform(X_dev)

model4 = LogisticRegression()
model4.fit(remove_top_train, y_train)
model4_dev_predictions = model4.predict(remove_top_dev)
result4 = accuracy_score(model4_dev_predictions, y_dev)
print(result4)

/ix1/cs1671-2026f/class-env/.pixi/envs/default/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


0.7165094339622642


In [13]:
#combining one word and removing prep word features
custom_train_features = hstack([one_word_train, remove_prep_train])
custom_dev_features = hstack([one_word_dev, remove_prep_dev])

model5 = LogisticRegression(max_iter=1000)
model5.fit(custom_train_features, y_train)
model5_dev_predictions = model5.predict(custom_dev_features)
result5 = accuracy_score(model5_dev_predictions, y_dev)
print(result5)

0.7323113207547169


In [14]:
#tesing the baseline (tokenizing based on ingredient)
model1_test_predictions = model1.predict(test_features)
test1 = pd.DataFrame(classification_report(y_test, model1_test_predictions, output_dict=True))
pd.set_option("display.max_columns", None)
print(test1)

           brazilian     british  cajun_creole     chinese    filipino  \
precision   0.789474    0.543860      0.747525    0.760532    0.712500   
recall      0.379747    0.226277      0.634454    0.812796    0.508929   
f1-score    0.512821    0.319588      0.686364    0.785796    0.593750   
support    79.000000  137.000000    238.000000  422.000000  112.000000   

               french       greek      indian      irish      italian  \
precision    0.536364    0.777027    0.829787   0.515625     0.748736   
recall       0.585608    0.646067    0.860927   0.362637     0.871429   
f1-score     0.559905    0.705521    0.845070   0.425806     0.805437   
support    403.000000  178.000000  453.000000  91.000000  1190.000000   

            jamaican    japanese      korean     mexican    moroccan  \
precision   0.820000    0.765027    0.793103    0.859073    0.752809   
recall      0.561644    0.606061    0.661871    0.911885    0.638095   
f1-score    0.666667    0.676329    0.721569   

In [15]:
#full confusion matrix for the baseline model
y_true = y_test
y_pred = model1_test_predictions
pred = [f"pred_{name}" for name in model1.classes_]
true = [f"true_{name}" for name in model1.classes_]
confusion = pd.DataFrame(confusion_matrix(y_true, y_pred), columns=pred, index=true)
print(confusion)

                   pred_brazilian  pred_british  pred_cajun_creole  \
true_brazilian                 30             1                  1   
true_british                    0            31                  2   
true_cajun_creole               1             1                151   
true_chinese                    0             1                  1   
true_filipino                   1             0                  0   
true_french                     0             3                  4   
true_greek                      0             0                  1   
true_indian                     1             1                  0   
true_irish                      0             7                  0   
true_italian                    1             1                 10   
true_jamaican                   1             0                  2   
true_japanese                   0             0                  0   
true_korean                     0             0                  0   
true_mexican        

In [16]:
#confusion matrix for each cuisine of baseline model
#1 = cuisine named above table
#0 = other cuisine
#rows = actual label
#columns = model's predicted label
print(len(model1.classes_))
mcm = multilabel_confusion_matrix(y_test, model1_test_predictions, labels=model1.classes_)
total_precision = 0
for i, name in zip(mcm, model1.classes_):
    total_precision += (i[1,1]/sum(i[:,1]))
    print(name)
    print(pd.DataFrame(i, columns=['pred_0', 'pred_1'], index=['true_0','true_1']))
    print('\n')

20
brazilian
        pred_0  pred_1
true_0    5970       8
true_1      49      30


british
        pred_0  pred_1
true_0    5894      26
true_1     106      31


cajun_creole
        pred_0  pred_1
true_0    5768      51
true_1      87     151


chinese
        pred_0  pred_1
true_0    5527     108
true_1      79     343


filipino
        pred_0  pred_1
true_0    5922      23
true_1      55      57


french
        pred_0  pred_1
true_0    5450     204
true_1     167     236


greek
        pred_0  pred_1
true_0    5846      33
true_1      63     115


indian
        pred_0  pred_1
true_0    5524      80
true_1      63     390


irish
        pred_0  pred_1
true_0    5935      31
true_1      58      33


italian
        pred_0  pred_1
true_0    4519     348
true_1     153    1037


jamaican
        pred_0  pred_1
true_0    5975       9
true_1      32      41


japanese
        pred_0  pred_1
true_0    5783      43
true_1      91     140


korean
        pred_0  pred_1
true_0    5894 

In [17]:
#pooled confusion matrix
pooled = pd.DataFrame(mcm.sum(axis=0), columns=['pred_0', 'pred_1'], index=['true_0', 'true_1'])
print(pooled)
#microaveraged precision
print("microaveraged precision: " + str(pooled.loc['true_1', 'pred_1']/sum(pooled['pred_1'])))
#macroaveraged precision
print("macroaveraged precision: " + str(total_precision/len(model1.classes_)))

        pred_0  pred_1
true_0  113473    1610
true_1    1610    4447
microaveraged precision: 0.7341918441472676
macroaveraged precision: 0.7032769495103058


In [18]:
#top 5 most informative features (leaning toward the cuisine)
feature_names = vectorizer.get_feature_names_out().tolist()
def most_informative_features(feature_names, classifier, cuisine, n=5):
    topn_names = []
    assert len(feature_names) == classifier.coef_.shape[1]
    k = list(classifier.classes_).index(cuisine)
    topn = sorted(zip(classifier.coef_[k], feature_names))[-n:]
    for coef, feat in topn:
        print(feat, coef)
        topn_names.append(feat)
    return topn_names
clf = model1 # FILL IN the classifier you want to examine
for name in model1.classes_:
    print(name)
    cuisine = name # FILL in the class ID you want to examine
    truth_names = most_informative_features(feature_names, clf, cuisine)
    print(truth_names)
    print('\n')

brazilian
açai 1.604439093897315
lime 1.762971731728339
dried black beans 2.0858765803051513
sweetened condensed milk 2.1821048024043983
cachaca 2.843595930259465
['açai', 'lime', 'dried black beans', 'sweetened condensed milk', 'cachaca']


british
pitted date 1.4933319061224823
stilton cheese 1.5373808222401415
caster sugar 1.6865776223949678
dried currants 1.76335688755319
stilton 2.0208914027980884
['pitted date', 'stilton cheese', 'caster sugar', 'dried currants', 'stilton']


cajun_creole
celery ribs 1.796197057815902
french bread 2.015985481445029
smoked sausage 2.0374233058377036
creole seasoning 3.1888894269160857
cajun seasoning 3.486644241355188
['celery ribs', 'french bread', 'smoked sausage', 'creole seasoning', 'cajun seasoning']


chinese
corn starch 1.8063234281261475
szechwan peppercorns 1.884846879918235
peanut oil 1.9878693667360359
soy sauce 2.080685822120289
chinese five-spice powder 2.511442536334324
['corn starch', 'szechwan peppercorns', 'peanut oil', 'soy sauce